# Rainfall–runoff event identification and association

This notebook demonstrates the rainfall–runoff event workflow implemented in `hydroevents`.

The workflow identifies rainfall events, detects runoff events from the stormflow series, associates rainfall and runoff events using physical constraints, and returns clean event-scale metrics.

## Method overview

The workflow includes:

1. conversion of stormflow discharge to runoff depth, if needed;
2. runoff event identification from the stormflow series;
3. rainfall event identification from the precipitation series;
4. rainfall–runoff event association using timing, lag and runoff-ratio constraints;
5. temporal adjustment of rainfall and runoff windows through backfilling and forward extension when required;
6. event filtering and quality control based on runoff coefficient, duration and lag;
7. event-scale linear reservoir fitting.

The final output is a clean table of associated rainfall–runoff events.

## Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import hydroevents

from hydroevents import rainfall_runoff_event

## Load input data

In [ ]:
DATA_DIR = Path("../examples/data")

runoff_file = DATA_DIR / "baseflow_separation.xlsx"
rain_file = DATA_DIR / "P_example.xlsx"

df_runoff = pd.read_excel(runoff_file)
df_rain = pd.read_excel(rain_file)

df_runoff["Date"] = pd.to_datetime(df_runoff["Date"])
df_rain["Date"] = pd.to_datetime(df_rain["Date"])

df_runoff = df_runoff.set_index("Date")
df_rain = df_rain.set_index("Date")

## Rainfall–runoff event identification

The procedure consists of four main steps:

1. **Runoff event identification**
   - Runoff events are extracted from the stormflow series (`Stormflow_mm`).
   - A noise threshold is automatically estimated from the lower part of the positive stormflow distribution and used to distinguish meaningful stormflow responses from residual numerical noise.
   - Runoff peaks are detected using a prominence-based approach.
   - Continuous periods of stormflow above the noise threshold are identified as candidate runoff events.
   - Multi-peak runoff responses are analysed and, when appropriate, separated into independent runoff events according to internal low-flow and peak-separation criteria.

2. **Rainfall event identification**
   - Rainfall events are extracted from the precipitation series using a precipitation threshold (`TOL_P`).
   - Consecutive rainfall periods separated by less than a user-defined dry interval are merged into a single rainfall event.
   - Events with cumulative rainfall below a minimum threshold are discarded.

3. **Rainfall–runoff association**
   - Each runoff event is associated with the most relevant antecedent rainfall event.
   - Associations are constrained by temporal consistency, including maximum rainfall–runoff lag and runoff coefficient limits.
   - Additional backfill procedures are used to capture rainfall occurring immediately before the initially selected rainfall event when hydrologically justified.

4. **Final event filtering**
   - Events with unrealistic runoff coefficients, excessive duration or anomalous rainfall–runoff lag are removed.
   - The remaining events are retained as the final set of rainfall–runoff events available for subsequent analyses.

For each identified event, the workflow returns rainfall and runoff start/end times, rainfall and runoff volumes, runoff peaks, event duration, runoff coefficient, and several diagnostic metrics describing the rainfall–runoff response.

## Define event-detection parameters

The parameters are grouped below according to the step in which they are used.
The workflow uses the following parameters:

1. **Runoff event identification**
These parameters control the extraction of runoff events from the stormflow series.
- `MIN_PROMINENCE` *(mm)*: minimum prominence used for runoff peak detection;
- `MIN_DISTANCE` *(time steps)*: minimum distance between runoff peaks;
- `WIDTH` *(time steps, optional)*: optional minimum peak width used during runoff peak detection. If set to `None`, no width constraint is applied.

2. **Rainfall event identification**
These parameters control the extraction of rainfall events from the precipitation series.
- `TOL_P` *(mm/h)*: precipitation threshold used to identify rainy time steps;
- `DRY_INTERVAL_HOURS` *(hours)*: minimum dry interval used to separate rainfall events;
- `MIN_CUMULATIVE_P` *(mm)*: minimum rainfall volume required for a rainfall event;

3. **Rainfall–runoff association**
These parameters control the matching between rainfall and runoff events.
- `MAX_LAG_H` *(hours)*: maximum allowed lag between rainfall peak and runoff peak;
- `RAIN_KEEP_FRAC` *(-)*: relative rainfall-volume threshold used to retain relevant rainfall candidates;
- `BACKFILL_HOURS` *(hours)*: maximum backward search window used when the runoff event starts before the assigned rainfall event. Within this window, previously unassigned rainfall can be added to the event, and the runoff start is adjusted to the local stormflow minimum between the updated rainfall start and the original runoff start.
- `RUNOFF_RATIO_LIMIT` *(-)*: maximum allowed ratio between runoff volume and rainfall volume during rainfall–runoff association;

4. **Final event filtering**
These parameters are applied after rainfall–runoff association to remove unrealistic events.
- `MAX_DURATION_H` *(hours)*: maximum total rainfall–runoff event duration. If set to None, a default value of 360 hours is used;
- `FINAL_RC_LIMIT` *(-)*: maximum accepted runoff coefficient during final filtering.

Tested default values for hourly rainfall–runoff analyses are:

**Runoff event identification**
- `MIN_PROMINENCE = 0.001`
- `MIN_DISTANCE = 3`
- `WIDTH = None`

**Rainfall event identification**
- `TOL_P = 0.1`
- `DRY_INTERVAL_HOURS = 6`
- `MIN_CUMULATIVE_P = 5`

**Rainfall–runoff association**
- `MAX_LAG_H = 120`
- `RAIN_KEEP_FRAC = 0.20`
- `RUNOFF_RATIO_LIMIT = 0.90`

**Final event filtering**
- `MAX_DURATION_H = 360`
- `FINAL_RC_LIMIT = 0.95`
  
Users can modify these parameters according to the the hydrological characteristics of the study catchment.

In [3]:
MIN_PROMINENCE = 0.001
MIN_DISTANCE = 3
WIDTH = None 

TOL_P = 0.1
DRY_INTERVAL_HOURS = 6
MIN_CUMULATIVE_P = 5

MAX_LAG_H = 120
RAIN_KEEP_FRAC = 0.20
RUNOFF_RATIO_LIMIT = 0.90
BACKFILL_HOURS = 6

MAX_DURATION_H = 360
FINAL_RC_LIMIT = 0.90

## Run workflow

In [2]:
event_results = rainfall_runoff_event(
    df_runoff=df_runoff,
    df_rain=df_rain,
    basin_name="example_basin",
    area_km2=150,
    stormflow_col="Stormflow",
    tol_P=TOL_P,
    dry_interval_hours=DRY_INTERVAL_HOURS,
    min_cumulative_P=MIN_CUMULATIVE_P,
    min_prominence=MIN_PROMINENCE,
    min_distance=MIN_DISTANCE,
    width=WIDTH,
    max_lag_h=MAX_LAG_H,
    max_duration_h=MAX_DURATION_H,
    rain_keep_frac=RAIN_KEEP_FRAC,
    runoff_ratio_limit=RUNOFF_RATIO_LIMIT,
    backfill_hours=BACKFILL_HOURS,
    final_rc_limit=FINAL_RC_LIMIT,
)

NameError: name 'rainfall_runoff_event' is not defined

## Event statistics

In [ ]:
print("Rainfall events:", len(event_results["rain_events"]))
print("Runoff events:", len(event_results["runoff_events"]))
print("Associated events:", len(event_results["associated_events"]))
print("Clean events:", len(event_results["clean_events"]))
print("Discarded events:", len(event_results["discarded_events"]))
print("Final outliers:", len(event_results["final_outliers"]))

## Clean rainfall–runoff events

The clean event table contains the final associated rainfall–runoff events after temporal consistency checks and quality-control filtering.

In [ ]:
clean_events = event_results["clean_events"]
clean_events.head()

## Runoff coefficient and linear reservoir parameters

For each associated rainfall–runoff event, the workflow computes the event runoff coefficient:

```text
RC = Runoff_Volume_mm / Rain_Volume_mm
```

The runoff coefficient represents the fraction of rainfall volume converted into direct runoff during the event.

Events with physically unrealistic runoff coefficients are filtered during both the rainfall–runoff association phase and the final quality-control stage. In particular, events with `RC` outside the range 0–1, or above the selected `FINAL_RC_LIMIT`, are discarded.

## Event-scale linear reservoir fitting

For each final rainfall–runoff event, HydroEvents can fit a simple linear reservoir model to the observed stormflow response.

The model assumes that a fixed fraction of rainfall contributes to runoff generation (`RC_lr`) and that the resulting runoff is routed through a linear reservoir characterized by a storage coefficient (`kd_lr_h`).

Model parameters are estimated independently for each event by minimizing the root mean square error (RMSE) between observed and simulated stormflow.

The fitting procedure returns:

- `RC_lr`: event-scale runoff coefficient estimated from the linear reservoir model;
- `kd_lr_h`: calibrated reservoir storage coefficient (hours);
- `RMSE_lr`: goodness-of-fit metric between observed and simulated stormflow.

In [ ]:
clean_events[
    [
        "Rain_Volume_mm",
        "Runoff_Volume_mm",
        "RC",
        "RC_lr",
        "kd_lr_h",
        "RMSE_lr",
    ]
].head()

## Plot rainfall and runoff events

In [ ]:
rain_events = event_results["rain_events"]
runoff_events = event_results["runoff_events"]
clean_events = event_results["clean_events"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_runoff.index,
        y=df_runoff["Stormflow"],
        mode="lines",
        name="Stormflow",
    )
)

fig.add_trace(
    go.Bar(
        x=df_rain.index,
        y=df_rain["P"],
        name="Precipitation",
        yaxis="y2",
        opacity=0.4,
    )
)

for _, row in clean_events.iterrows():
    fig.add_vrect(
        x0=row["Rain_Start"],
        x1=row["Rain_End"],
        fillcolor="LightBlue",
        opacity=0.25,
        line_width=0,
    )

    fig.add_vrect(
        x0=row["Runoff_Start"],
        x1=row["Runoff_End"],
        fillcolor="LightGreen",
        opacity=0.20,
        line_width=0,
    )

fig.update_layout(
    title="Associated rainfall–runoff events",
    xaxis_title="Date",
    yaxis=dict(title="Stormflow (m³/s)"),
    yaxis2=dict(
        title="Precipitation",
        overlaying="y",
        side="right",
        autorange="reversed",
    ),
    template="plotly_white",
)

fig.show()

## Plot Rainfall volume vs runoff volume

In [ ]:
# Rainfall volume vs runoff volume
if not clean_events.empty:

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(
        clean_events["Rain_Volume_mm"],
        clean_events["Runoff_Volume_mm"],
    )

    ax.set_xlabel("Rainfall volume (mm)")
    ax.set_ylabel("Runoff volume (mm)")
    ax.set_title("Rainfall vs runoff event volumes")
    ax.grid(alpha=0.3)

    plt.show()

## Plot Rainfall volume vs observed and simulated runoff coefficients

- `RMSE_LR_THRESHOLD`: threshold used to highlight events with low linear-reservoir fitting error in diagnostic plots. This parameter does not affect event identification or filtering.

In [ ]:
RMSE_LR_THRESHOLD = 0.1

# Runoff coefficient vs rainfall volume
if not clean_events.empty:

    rmse_threshold = RMSE_LR_THRESHOLD

    df_rc = clean_events.copy()

    valid_sim = (
        df_rc["RC_lr"].notna()
        & df_rc["RMSE_lr"].notna()
        & (df_rc["RMSE_lr"] <= rmse_threshold)
    )

    bad_sim = (
        df_rc["RC_lr"].notna()
        & df_rc["RMSE_lr"].notna()
        & (df_rc["RMSE_lr"] > rmse_threshold)
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    # observed RC
    ax.scatter(
        df_rc["Rain_Volume_mm"],
        df_rc["RC"],
        alpha=0.7,
        marker="^",
        label="Observed RC",
    )

    # good simulations
    ax.scatter(
        df_rc.loc[valid_sim, "Rain_Volume_mm"],
        df_rc.loc[valid_sim, "RC_lr"],
        alpha=0.7,
        marker="X",
        label=f"Simulated RC | RMSE ≤ {rmse_threshold}",
    )

    # poor simulations
    ax.scatter(
        df_rc.loc[bad_sim, "Rain_Volume_mm"],
        df_rc.loc[bad_sim, "RC_lr"],
        alpha=0.25,
        marker="x",
        label=f"Simulated RC | RMSE > {rmse_threshold}",
    )

    ax.set_xlabel("Rainfall volume (mm)")
    ax.set_ylabel("Runoff coefficient (-)")
    ax.set_title("Runoff coefficient vs rainfall volume")

    ax.set_ylim(0, 1)

    ax.grid(alpha=0.3)

    ax.legend(frameon=False)

    plt.tight_layout()
    plt.show()

## Save output

The event tables are exported for subsequent event-based analyses.

In [ ]:
output_dir = Path("../examples/output")
output_dir.mkdir(parents=True, exist_ok=True)

event_results["rain_events"].to_excel(output_dir / "rain_events.xlsx", index=False)
event_results["runoff_events"].to_excel(output_dir / "runoff_events.xlsx", index=False)
event_results["associated_events"].to_excel(output_dir / "associated_events.xlsx", index=False)
event_results["discarded_events"].to_excel(output_dir / "discarded_events.xlsx", index=False)
event_results["clean_events"].to_excel(output_dir / "clean_events.xlsx", index=True)
event_results["final_outliers"].to_excel(output_dir / "final_outliers.xlsx", index=False)

print(f"Results saved to: {output_dir.resolve()}")